# NicheRadar Lymph Node Workflow

This notebook is the standard reproducible workflow for running NicheRadar on the lymph node spatial transcriptomics dataset. It uses the paper-aligned package name, class names, and AnnData output keys.


In [ ]:
# Setup
import ctypes
import logging
import os
import sys
import tempfile
import time
import tracemalloc
import warnings
from pathlib import Path


def preload_nvjitlink():
    """Load nvJitLink before packages that may import torch in CUDA builds."""

    candidates = []
    if os.environ.get("CONDA_PREFIX"):
        candidates.append(Path(os.environ["CONDA_PREFIX"]))
    candidates.append(Path(sys.executable).resolve().parents[1])

    for prefix in candidates:
        for lib in (prefix / "lib").glob(
            "python*/site-packages/nvidia/nvjitlink/lib/libnvJitLink.so.12"
        ):
            if lib.exists():
                ctypes.CDLL(str(lib), mode=ctypes.RTLD_GLOBAL)
                return lib
    return None


preloaded_nvjitlink = preload_nvjitlink()
warnings.filterwarnings("ignore", message="IProgress not found.*", category=Warning)
warnings.filterwarnings("ignore", module="tqdm.auto")
matplotlib_cache_dir = Path(tempfile.gettempdir()) / "nicheradar_matplotlib_cache"
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache_dir))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from scipy import sparse


def find_project_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "NicheRadar", cwd.parent / "NicheRadar"]
    for candidate in candidates:
        if (candidate / "nicheradar" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the NicheRadar project directory. Run this notebook from "
        "NicheRadar/, NicheRadar/notebooks/, or the parent directory containing NicheRadar/."
    )


PROJECT_DIR = find_project_dir()
sys.path.insert(0, str(PROJECT_DIR))

from nicheradar import NicheRadar

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
logger = logging.getLogger("NicheRadar-notebook")
logger.info("NicheRadar project directory resolved.")
if preloaded_nvjitlink is not None:
    logger.info("Optional CUDA support library preloaded.")


In [ ]:
# Configuration
QUICK_TEST = False
SAVE_PROCESSED_H5AD = False
MODULE_ACTIVITY_KEY = "normalized_expression"  # "normalized_expression" or "coverage"

MODEL_CONFIG = {
    "layer": None,
    "use_raw": False,
    "transform": "identity",
    "min_total_expression": 10.0,
    "min_mean_expression": 1e-4,
    "min_detected_spots": 20,
    "min_detected_fraction": 0.01,
    "min_variance": 1e-8,
    "n_svgs": 700,
    "svg_zscore_threshold": None,
    "svg_quantile": None,
    "chunk_size": 512,
    "n_gene_modules": 18,
    "hyperedge_local_k": 13,
    "hyperedge_n_iter": 400,
    "hyperedge_learning_rate": 5e-3,
    "hyperedge_diversity_weight": 1e-2,
    "enable_unassigned_hyperedge": False,
    "hyperedge_device": "cpu",
    "module_vote_rate": 0.15,
    "random_state": 303,
    "show_progress": True,
}

if QUICK_TEST:
    MODEL_CONFIG.update(
        {
            "n_svgs": 80,
            "n_gene_modules": 6,
            "hyperedge_local_k": 8,
            "hyperedge_n_iter": 25,
            "chunk_size": 256,
            "show_progress": False,
        }
    )

OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load data

def resolve_data_path(project_dir):
    env_path = os.environ.get("NICHERADAR_LYMPH_NODE_H5AD")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    candidates.extend(
        [
            project_dir / "data" / "lymph_node_niche_annotated.h5ad",
            project_dir / "data" / "lymph_node_niche_annotated copy.h5ad",
        ]
    )
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "Lymph node AnnData file not found. Put it at "
        "data/lymph_node_niche_annotated.h5ad relative to the project root "
        "or set NICHERADAR_LYMPH_NODE_H5AD."
    )


def subset_for_quick_test(adata, n_obs=1500, n_vars=500, seed=303):
    rng = np.random.default_rng(seed)
    result = adata
    if result.n_obs > n_obs:
        obs_idx = np.sort(rng.choice(result.n_obs, size=n_obs, replace=False))
        result = result[obs_idx].copy()
    else:
        result = result.copy()

    if result.n_vars > n_vars:
        X = result.X
        if sparse.issparse(X):
            totals = np.asarray(X.sum(axis=0)).ravel()
        else:
            totals = np.asarray(X, dtype=float).sum(axis=0)
        var_idx = np.sort(np.argsort(totals)[::-1][:n_vars])
        result = result[:, var_idx].copy()
    return result


DATA_PATH = resolve_data_path(PROJECT_DIR)
logger.info("Loading lymph node AnnData.")
adata = ad.read_h5ad(DATA_PATH)

if QUICK_TEST:
    adata = subset_for_quick_test(adata)

if "spatial" not in adata.obsm and {"x", "y"}.issubset(adata.obs.columns):
    adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy()


In [ ]:
# Visualization helpers

NICHERADAR_ACTIVITY_CMAP = LinearSegmentedColormap.from_list(
    "nicheradar_activity",
    ["#000000", "#B5162F", "#FFF04A"],
    N=256,
)


def spatial_coordinates(adata):
    if "spatial" in adata.obsm:
        coords = np.asarray(adata.obsm["spatial"], dtype=float)
    else:
        coords = adata.obs[["x", "y"]].to_numpy(dtype=float)
    return coords[:, :2]


def adaptive_point_size(n_obs):
    return float(np.clip(120000 / max(n_obs, 1), 4, 28))


def finish_spatial_axis(ax, title, invert_y=False):
    ax.set_title(title)
    ax.set_xlabel("spatial1")
    ax.set_ylabel("spatial2")
    ax.set_aspect("equal", adjustable="box")
    if invert_y:
        ax.invert_yaxis()
    ax.grid(False)


def plot_spatial_continuous(
    ax,
    coords,
    values,
    title,
    point_size,
    cmap=NICHERADAR_ACTIVITY_CMAP,
    invert_y=False,
):
    values = np.asarray(values, dtype=float)
    lo, hi = np.nanpercentile(values, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo = float(np.nanmin(values))
        hi = float(np.nanmax(values) + 1e-8)
    scatter = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=values,
        s=point_size,
        cmap=cmap,
        vmin=lo,
        vmax=hi,
        linewidths=0,
        alpha=0.95,
        rasterized=True,
    )
    finish_spatial_axis(ax, title, invert_y=invert_y)
    plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)


def plot_module_patterns(adata, model, pattern_score="normalized_expression", max_modules=18):
    coords = spatial_coordinates(adata)
    point_size = adaptive_point_size(adata.n_obs)
    module_ids = np.asarray(model.module_ids_, dtype=int)
    if pattern_score == "normalized_expression":
        scores = np.asarray(model.module_scores_, dtype=float)
        score_label = "Normalized expression"
    elif pattern_score == "coverage":
        scores = np.asarray(model.module_coverage_, dtype=float)
        score_label = "Coverage"
    else:
        raise ValueError("pattern_score must be 'normalized_expression' or 'coverage'.")

    table = model.module_table_.head(max_modules).copy()
    n_cols = min(3, max(1, len(table)))
    n_rows = int(np.ceil(len(table) / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5.2 * n_cols, 4.4 * n_rows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, table.iterrows()):
        module_id = int(row["module"])
        col = int(np.flatnonzero(module_ids == module_id)[0])
        title = f"{score_label} M{module_id} | genes={int(row['n_genes'])}"
        plot_spatial_continuous(ax, coords, scores[:, col], title, point_size)
    for ax in axes[len(table) :]:
        ax.axis("off")
    plt.show()


In [ ]:
# Run NicheRadar module discovery
model = NicheRadar(**MODEL_CONFIG)

tracemalloc.start()
start = time.time()
adata = model.fit_predict(adata, copy=False)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

run_stats = {
    "mode": "quick_test" if QUICK_TEST else "full",
    "seconds": time.time() - start,
    "memory_mb_current": current / 1e6,
    "memory_mb_peak": peak / 1e6,
    "n_obs": int(adata.n_obs),
    "n_vars": int(adata.n_vars),
    "n_svgs": int(model.svg_mask_.sum()),
    "n_gene_modules": int(model.n_gene_modules),
}

logger.info("Expression source: %s", adata.uns["nicheradar_expression_source"])
logger.info("Filter stats: %s", adata.uns["nicheradar_filter_stats"])
logger.info("Run stats: %s", run_stats)

display(pd.DataFrame([run_stats]))
display(model.module_table_)
display(model.svg_table_.head(20))

diagnostic_keys = [
    "hyperedge_initial_reconstruction_loss",
    "hyperedge_reconstruction_loss",
    "hyperedge_reconstruction_improvement",
    "hyperedge_initial_mean_max_membership",
    "hyperedge_mean_max_membership",
    "hyperedge_label_change_fraction_from_initial",
    "hyperedge_spatial_diversity_loss",
    "hyperedge_n_iter",
]
hyperedge_info = adata.uns["nicheradar_hyperedge_info"]
display(pd.DataFrame([{key: hyperedge_info.get(key) for key in diagnostic_keys}]))


In [ ]:
# Plot module activity maps
plot_module_patterns(
    adata,
    model,
    pattern_score=MODULE_ACTIVITY_KEY,
    max_modules=min(model.n_gene_modules, 18),
)


In [ ]:
# Niche annotation versus matched NicheRadar gene modules
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter

TRUTH_KEY = "niche"
if TRUTH_KEY not in adata.obs:
    raise KeyError(f"adata.obs[{TRUTH_KEY!r}] is required for the annotation-module comparison.")

TRUTH_DEFINITIONS = [
    {
        "label": "T cell Zone",
        "title": "T cell Zone (+ Medulla)",
        "mask_labels": ["T cell Zone", "Medulla"],
    },
    {
        "label": "B cell Zone",
        "title": "B cell Zone (+ GC)",
        "mask_labels": ["B cell Zone", "Germinal Center"],
    },
    {
        "label": "Medulla",
        "title": "Medulla",
        "mask_labels": ["Medulla"],
    },
    {
        "label": "Germinal Center",
        "title": "Germinal Center",
        "mask_labels": ["Germinal Center"],
    },
]

if MODULE_ACTIVITY_KEY == "normalized_expression":
    module_pattern_scores = np.asarray(model.module_scores_, dtype=float)
elif MODULE_ACTIVITY_KEY == "coverage":
    module_pattern_scores = np.asarray(model.module_coverage_, dtype=float)
else:
    raise ValueError("MODULE_ACTIVITY_KEY must be 'normalized_expression' or 'coverage'.")

module_ids = np.asarray(model.module_ids_, dtype=int)
truth_array = adata.obs[TRUTH_KEY].astype(str).to_numpy()
coords = spatial_coordinates(adata)


def binary_pattern_correlations(mask, scores):
    x = mask.astype(float)
    y = np.asarray(scores, dtype=float)
    x = x - x.mean()
    y = y - y.mean(axis=0, keepdims=True)
    denom = np.sqrt(np.sum(x ** 2) * np.sum(y ** 2, axis=0))
    return np.divide(x @ y, denom, out=np.full(y.shape[1], np.nan), where=denom > 0)


match_rows = []
for item in TRUTH_DEFINITIONS:
    mask = np.isin(truth_array, item["mask_labels"])
    if not np.any(mask):
        raise ValueError(f"No observations found for labels: {item['mask_labels']}")
    corr = binary_pattern_correlations(mask, module_pattern_scores)
    best_col = int(np.nanargmax(corr))
    match_rows.append(
        {
            "niche": item["label"],
            "title": item["title"],
            "mask_labels": item["mask_labels"],
            "mask_label_text": ", ".join(item["mask_labels"]),
            "matched_module": int(module_ids[best_col]),
            "module_col": best_col,
            "pearson_r": float(corr[best_col]),
            "n_spots": int(mask.sum()),
        }
    )

match_table = pd.DataFrame(match_rows)
display(match_table[["niche", "mask_label_text", "matched_module", "pearson_r", "n_spots"]])

niche_colors = {
    "T cell Zone": "#5B8FF9",
    "B cell Zone": "#F6BD16",
    "Medulla": "#9270CA",
    "Germinal Center": "#E86452",
}
module_cmap = NICHERADAR_ACTIVITY_CMAP
figure_path = OUTPUT_DIR / "nicheradar_niche_annotation_vs_gene_modules.png"

x_min, x_max = np.nanmin(coords[:, 0]), np.nanmax(coords[:, 0])
y_min, y_max = np.nanmin(coords[:, 1]), np.nanmax(coords[:, 1])
x_pad = 0.015 * (x_max - x_min)
y_pad = 0.030 * (y_max - y_min)

fig = plt.figure(figsize=(7.65, 6.35), facecolor="white")
grid = fig.add_gridspec(
    nrows=len(TRUTH_DEFINITIONS),
    ncols=4,
    width_ratios=[1.0, 1.0, 0.055, 0.035],
    wspace=0.075,
    hspace=0.135,
)
axes = np.empty((len(TRUTH_DEFINITIONS), 2), dtype=object)
for row_idx in range(len(TRUTH_DEFINITIONS)):
    axes[row_idx, 0] = fig.add_subplot(grid[row_idx, 0])
    axes[row_idx, 1] = fig.add_subplot(grid[row_idx, 1])
fig.add_subplot(grid[:, 2]).axis("off")
cax = fig.add_subplot(grid[:, 3])

background_color = "#D8DDE2"
background_alpha = 0.25
truth_size = 3
module_size = 1.3

for row_idx, row in match_table.iterrows():
    label = row["niche"]
    mask = np.isin(truth_array, row["mask_labels"])
    ax_truth = axes[row_idx, 0]
    ax_module = axes[row_idx, 1]

    ax_truth.scatter(
        coords[~mask, 0],
        coords[~mask, 1],
        s=truth_size,
        c=background_color,
        alpha=background_alpha,
        linewidths=0,
        rasterized=True,
    )
    ax_truth.scatter(
        coords[mask, 0],
        coords[mask, 1],
        s=truth_size,
        c=niche_colors[label],
        alpha=0.98,
        linewidths=0,
        rasterized=True,
    )
    ax_truth.set_title(row["title"], pad=2.0, fontweight="normal", fontsize=15)

    raw_score = np.asarray(module_pattern_scores[:, int(row["module_col"])], dtype=float)
    lo, hi = np.nanpercentile(raw_score, [1, 99.5])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        lo = float(np.nanmin(raw_score))
        hi = float(np.nanmax(raw_score) + 1e-8)
    score = np.clip(raw_score, lo, hi)
    score = (score - lo) / max(hi - lo, 1e-12)
    ax_module.scatter(
        coords[:, 0],
        coords[:, 1],
        c=score,
        s=module_size,
        cmap=module_cmap,
        vmin=0,
        vmax=1,
        linewidths=0,
        alpha=0.98,
        rasterized=True,
    )
    ax_module.set_title(f"Module {int(row['matched_module'])}", pad=2.0, fontweight="normal", fontsize=15)

    for ax in (ax_truth, ax_module):
        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_facecolor("white")
        for spine in ax.spines.values():
            spine.set_visible(False)

sm = ScalarMappable(cmap=module_cmap)
sm.set_clim(0, 1)
cbar = fig.colorbar(sm, cax=cax)
cbar.set_ticks([0, 0.5, 1.0])
cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
cbar.ax.tick_params(labelsize=10, length=2, width=0.6, pad=1.5)
cbar.outline.set_linewidth(0.5)

fig.savefig(figure_path, dpi=500, bbox_inches="tight", pad_inches=0.02, facecolor="white")
plt.show()
try:
    display_figure_path = figure_path.relative_to(PROJECT_DIR)
except ValueError:
    display_figure_path = figure_path.name
print(f"Saved annotation-module comparison figure to: {display_figure_path}")
